# No-Search HMDB Evaluation Workflow

## 1. Set Up Environment Variables and Imports
Load configuration, environment variables, and runtime dependencies for the no-search agent.

In [1]:
import os
import sys
import json
import re
import logging
import time
import copy
import uuid
from datetime import datetime
from pathlib import Path
from typing import Optional, List, TypedDict

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.utils.utils import secret_from_env
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError
from pydantic import Field, SecretStr
from tqdm.auto import tqdm

load_dotenv()

PROJECT_ROOT = Path("/home/chen/proj/db_form_agent_system")
PROMPTS_DIR = PROJECT_ROOT / "prompts"
RESULTS_DIR = PROJECT_ROOT / "no_search_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT / "code"))

class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key", default_factory=secret_from_env("OPENROUTER_API_KEY", default=None)
    )

    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self, openai_api_key: Optional[str] = None, **kwargs):
        openai_api_key = openai_api_key or os.environ.get("OPENROUTER_API_KEY")
        super().__init__(base_url="https://openrouter.ai/api/v1", openai_api_key=openai_api_key, **kwargs)


no_search_model = ChatOpenRouter(model_name="google/gemini-2.5-flash", temperature=0.0)
critique_model = ChatOpenRouter(model_name="openai/gpt-4o-mini", temperature=0.0)

MAX_ITERATIONS = 20

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(name)s | %(message)s")

def _vprint(*args, **kwargs):
    if os.getenv("NODE_LOGS", "1") in ("1", "true", "TRUE", "yes", "YES"):
        print(*args, **kwargs)



/home/chen/proj/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load HMDB Dataset Samples
Read the HMDB metabolite catalog, stratify endogenous/exogenous entries, and prepare a balanced evaluation subset.

In [2]:
HMDB_SOURCE = Path("/data/chen/data/hmdb/metabolites.csv")
if not HMDB_SOURCE.exists():
    raise FileNotFoundError(f"Expected HMDB dataset at {HMDB_SOURCE}")

df_hmdb = pd.read_csv(HMDB_SOURCE)
df_endogenous = df_hmdb[df_hmdb["Endogenous"] == "Yes"].sample(n=45, random_state=42)
df_exogenous = df_hmdb[df_hmdb["Endogenous"] == "No"].sample(n=5, random_state=42)
df_sampled = (
    pd.concat([df_endogenous, df_exogenous], ignore_index=True)
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
 )

print(f"Total sampled compounds: {len(df_sampled)}")
df_sampled.head()


Total sampled compounds: 50


,Accession,Name,Description,Chemical_Formula,Average_Molecular_Weight,Monisotopic_Molecular_Weight,IUPAC_Name,Traditional_IUPAC,CAS_Registry_Number,SMILES,InChI,InChIKey,Synonyms,Ontology,Endogenous
0,HMDB0013450,"PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))","PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z)) is a ph...",C52H94NO7P,876.2790,875.676791,"(2-{[(2R)-2-[(4Z,7Z,10Z,13Z,16Z,19Z)-docosa-4,...","(2-{[(2R)-2-[(4Z,7Z,10Z,13Z,16Z,19Z)-docosa-4,...",NaN,CCCCCCCCCCCCCCCCCCCCCCOC[C@]([H])(COP([O-])(=O...,InChI=1S/C52H94NO7P/c1-6-8-10-12-14-16-18-20-2...,YLLDQHGIGWZVQP-ITBHSSNISA-N,1-Behenyl-2-docosahexaenoyl-sn-glycero-3-phosp...,\n,Yes
1,HMDB0223787,CL(14:0/i-14:0/18:0/23:0),CL(14:0/i-14:0/18:0/23:0) is a cardiolipin (CL...,C78H152O17P2,1424.0050,1423.050477,[(2R)-2-hydroxy-3-({hydroxy[(2R)-3-(octadecano...,(2R)-2-hydroxy-3-{[hydroxy((2R)-3-(octadecanoy...,NaN,[H][C@@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCC...,InChI=1S/C78H152O17P2/c1-6-9-12-15-18-21-24-26...,QJGGXVYKMFFNAP-FXCIZWNUSA-N,[(2R)-2-Hydroxy-3-({hydroxy[(2R)-3-(octadecano...,\n,Yes
2,HMDB0068522,TG(22:0/i-12:0/i-24:0),TG(22:0/i-12:0/i-24:0) belongs to the family o...,C61H118O6,947.6090,946.892842,(2S)-3-(docosanoyloxy)-2-[(10-methylundecanoyl...,(2S)-3-(docosanoyloxy)-2-[(10-methylundecanoyl...,NaN,[H][C@](COC(=O)CCCCCCCCCCCCCCCCCCCCC)(COC(=O)C...,InChI=1S/C61H118O6/c1-6-7-8-9-10-11-12-13-14-1...,JAWPDRHZOXOVGJ-XKJQNMSGSA-N,1-behenoyl-2-isododecanoyl-3-isotetracosanoyl-...,\n,Yes
3,HMDB0001388,alpha-Linolenic acid,alpha-Linolenic acid (ALA) is a polyunsaturate...,C18H30O2,278.4296,278.224580,"(9Z,12Z,15Z)-octadeca-9,12,15-trienoic acid",α linolenic acid,463-40-1,CC\C=C/C\C=C/C\C=C/CCCCCCCC(O)=O,InChI=1S/C18H30O2/c1-2-3-4-5-6-7-8-9-10-11-12-...,DTOSIQBPPRVQHS-PDBXOOCHSA-N,"(9,12,15)-Linolenic acid; (9Z,12Z,15Z)-Octadec...",\n,No
4,HMDB0219059,CL(a-13:0/i-17:0/18:0/22:0),CL(a-13:0/i-17:0/18:0/22:0) is a cardiolipin (...,C79H154O17P2,1438.0320,1437.066127,[(2R)-3-({[(2R)-2-(docosanoyloxy)-3-(octadecan...,(2R)-3-{[(2R)-2-(docosanoyloxy)-3-(octadecanoy...,NaN,[H][C@@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCC...,InChI=1S/C79H154O17P2/c1-7-10-12-14-16-18-20-2...,BFRUSIFRUHYVKZ-QDFPNRTDSA-N,[(2R)-3-({[(2R)-2-(docosanoyloxy)-3-(octadecan...,\n,Yes


## 3. Assemble No-Search Prompt Templates
Load dedicated system prompts that constrain the agent to rely purely on internal knowledge.

In [3]:
NO_SEARCH_PROMPT = SystemMessage(
    content=(PROMPTS_DIR / "search_only_agent.md").read_text()
 )
CRITIQUE_NO_SEARCH_PROMPT = SystemMessage(
    content=(PROMPTS_DIR / "critique_search_only_agent.md").read_text()
 )

print("Loaded no-search prompt templates.")



Loaded no-search prompt templates.


## 4. Construct No-Search LangGraph Workflow
Build a LangGraph pipeline with a single no-search agent followed by the critique reviewer, omitting all retrieval nodes.

In [4]:
agent_tools: list = []  # No external tools allowed
memory = MemorySaver()

no_search_executor = create_react_agent(
    no_search_model,
    agent_tools,
    prompt=NO_SEARCH_PROMPT,
    checkpointer=memory,
 )
critique_executor = create_react_agent(
    critique_model,
    agent_tools,
    prompt=CRITIQUE_NO_SEARCH_PROMPT,
    checkpointer=memory,
 )

class StateHistory(TypedDict):
    claim: str
    confidence_score: float
    supporting_sources: List[str]
    tool_name: str | None
    stop_reason: str | None


class AgentState(TypedDict):
    messages: List[HumanMessage | AIMessage]
    original_query: str
    stop_history: List[StateHistory]
    consecutive_no_new_evidence: int
    consecutive_stagnant_claim: int
    consecutive_tool_unavailable: int
    last_critique_signature: str | None
    consecutive_clarify_without_progress: int


def _last_agent_json(messages: List[HumanMessage | AIMessage]) -> str:
    if not messages:
        return ""
    for msg in reversed(messages):
        content = getattr(msg, "content", "") or ""
        if '"synthesis"' in content:
            return content
    return ""


def determine_final_claim(messages: List[HumanMessage | AIMessage]) -> str:
    blob = _last_agent_json(messages)
    if not blob:
        return "Unclear"
    try:
        obj = json.loads(blob)
        synthesis = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        claim_raw = synthesis.get("claim") if isinstance(synthesis, dict) else None
        if isinstance(claim_raw, str) and claim_raw.strip():
            return claim_raw.strip()
    except Exception:
        pass
    match = re.search(r'"claim"\s*:\s*"([^"]+)"', blob)
    if match:
        candidate = match.group(1).strip()
        if candidate:
            return candidate
    return "Unclear"


def _safe_float(value):
    try:
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            return float(value.replace("–", "-").strip())
    except Exception:
        return None
    return None


def determine_confidence_score(messages: List[HumanMessage | AIMessage]) -> float:
    blob = _last_agent_json(messages)
    if not blob:
        return 0.0
    try:
        obj = json.loads(blob)
        synthesis = obj.get("synthesis", {}) if isinstance(obj, dict) else {}
        value = synthesis.get("confidence_score") if isinstance(synthesis, dict) else None
        candidate = _safe_float(value)
        if candidate is not None:
            return max(0.0, min(1.0, candidate))
    except Exception:
        pass
    match = re.search(r'"confidence_score"\s*:\s*("?)([0-9]*\.?[0-9]+)\1', blob)
    if match:
        candidate = _safe_float(match.group(2))
        if candidate is not None:
            return max(0.0, min(1.0, candidate))
    return 0.0


def no_search_node(state: AgentState):
    _vprint("-----No-Search Node Invoked-----")
    input_messages = state["messages"]
    last_content = state["messages"][-1].content if state.get("messages") else ""
    follow_up = None
    match = re.search(r"\{[\s\S]*\}", str(last_content))
    if match:
        try:
            payload = json.loads(match.group(0))
            if (payload.get("decision", "") or "").lower() == "clarify":
                follow_up = payload.get("next_query")
        except json.JSONDecodeError:
            pass
    if follow_up:
        merged = (
            f"Original question:\n{state.get('original_query', '')}\n\n"
            f"Reviewer follow-up:\n{follow_up}\n\n"
            "Respond using only remembered knowledge; do not claim new searches."
        )
        input_messages = [HumanMessage(content=merged)]
    result = no_search_executor.invoke({"messages": input_messages})
    return {"messages": state["messages"] + [AIMessage(content=result['messages'][-1].content)]}


def critique_node(state: AgentState):
    _vprint("-----Critique Node Invoked-----")
    last_message = state["messages"][-1].content if state.get("messages") else ""
    original = state.get("original_query", "")
    critique_input = f"Original question: {original}\nNo-search output:\n{last_message}"
    critique_result = critique_executor.invoke({"messages": [HumanMessage(content=critique_input)]})
    critique_content = critique_result['messages'][-1].content
    raw = (critique_content or "").strip()

    parsed: dict | None = None
    try:
        candidate = json.loads(raw)
        if isinstance(candidate, dict):
            parsed = candidate
    except Exception:
        pass
    if parsed is None:
        match = re.search(r"\{[\s\S]*?\}", raw)
        if match:
            try:
                candidate = json.loads(match.group(0))
                if isinstance(candidate, dict):
                    parsed = candidate
            except Exception:
                parsed = None

    stop_reason = None
    supporting_sources: List[str] = []
    tool_name: str | None = None

    if parsed:
        candidate_reason = parsed.get("stop_reason")
        if isinstance(candidate_reason, str) and candidate_reason.strip():
            stop_reason = candidate_reason.strip().upper()

    history = copy.deepcopy(state.get("stop_history", []))
    claim = determine_final_claim(state.get("messages", []))
    confidence = determine_confidence_score(state.get("messages", []))

    history.append({
        "claim": claim,
        "confidence_score": confidence,
        "supporting_sources": supporting_sources,
        "tool_name": tool_name,
        "stop_reason": stop_reason,
    })

    prev_no_new = state.get("consecutive_no_new_evidence", 0)
    prev_stagnant = state.get("consecutive_stagnant_claim", 0)
    prev_tool_unavailable = state.get("consecutive_tool_unavailable", 0)
    clarify_without_progress = state.get("consecutive_clarify_without_progress", 0)
    previous_signature = state.get("last_critique_signature") or ""

    raw_signature = json.dumps(parsed, sort_keys=True) if parsed else raw

    decision = parsed.get("decision") if parsed else None
    if isinstance(decision, str) and decision.lower() == "clarify" and not stop_reason:
        if raw_signature == previous_signature:
            clarify_without_progress += 1
        else:
            clarify_without_progress = 1
        if clarify_without_progress >= 3:
            stop_reason = "STAGNANT_CLAIM"
            prev_stagnant = max(prev_stagnant, clarify_without_progress)
    else:
        clarify_without_progress = 0

    no_new = prev_no_new
    if not stop_reason and isinstance(decision, str) and decision.lower() == "clarify":
        no_new = prev_no_new + 1 if raw_signature == previous_signature else 0

    if not stop_reason and (isinstance(decision, str) and decision.lower() == "clarify") and clarify_without_progress >= 2:
        stop_reason = "NO_NEW_EVIDENCE"
        history[-1]["stop_reason"] = stop_reason
        no_new = prev_no_new + 1

    if history:
        history[-1]["stop_reason"] = stop_reason

    patched_content = critique_content
    if parsed is not None:
        patched = dict(parsed)
        if stop_reason:
            patched["stop_reason"] = stop_reason
        patched_content = json.dumps(patched, indent=2)

    return {
        "messages": state["messages"] + [AIMessage(content=patched_content)],
        "stop_history": history,
        "consecutive_no_new_evidence": no_new,
        "consecutive_stagnant_claim": prev_stagnant,
        "consecutive_tool_unavailable": prev_tool_unavailable,
        "last_critique_signature": raw_signature,
        "consecutive_clarify_without_progress": clarify_without_progress,
    }


def should_continue(state: AgentState):
    last_message_content = state["messages"][-1].content if state.get("messages") else ""
    raw = (last_message_content or "").strip()
    _vprint(f"[should_continue] last critique output: {raw[:200]}...")

    STOP_REASONS = {"APPROVED", "NO_NEW_EVIDENCE", "STAGNANT_CLAIM", "TOOL_UNAVAILABLE", "FUTILITY_DETECTED"}

    history = state.get("stop_history") or []
    if history:
        latest_reason = history[-1].get("stop_reason")
        if isinstance(latest_reason, str) and latest_reason.strip().upper() in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (history stop_reason={latest_reason})-----")
            return END

    if state.get("consecutive_clarify_without_progress", 0) >= 5:
        _vprint("-----Agent Decided to Stop (clarify stagnation >=5)-----")
        return END

    def _normalized_reason(value):
        if isinstance(value, str) and value.strip():
            return value.strip().upper()
        return None

    def _should_stop_from_json(obj: dict) -> bool:
        stop_reason = _normalized_reason(obj.get("stop_reason"))
        if stop_reason in STOP_REASONS:
            _vprint(f"-----Agent Decided to Stop (stop_reason={stop_reason})-----")
            return True
        decision = obj.get("decision")
        if isinstance(decision, str) and decision.lower() == "approve":
            _vprint("-----Agent Decided to Stop (decision=approve)-----")
            return True
        try:
            score = float(obj.get("agreement_score", 0.0))
            if score > 0.75:
                _vprint(f"-----Agent Decided to Stop (legacy agreement_score={score:.2f} > 0.75)-----")
                return True
        except Exception:
            pass
        return False

    try:
        critique_json = json.loads(raw)
        if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
            return END
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*?\}", raw)
    if match:
        try:
            critique_json = json.loads(match.group(0))
            if isinstance(critique_json, dict) and _should_stop_from_json(critique_json):
                _vprint("-----Agent Decided to Stop (extracted JSON)-----")
                return END
        except Exception:
            pass

    if any(token in raw.lower() for token in ["approve", "approved"]):
        _vprint("-----Agent Decided to Stop (text match approve)-----")
        return END

    return "no_search"


app = None

def _ensure_app() -> None:
    global app
    if app is None:
        workflow = StateGraph(AgentState)
        workflow.add_node("no_search", no_search_node)
        workflow.add_node("critique", critique_node)
        workflow.set_entry_point("no_search")
        workflow.add_edge("no_search", "critique")
        workflow.add_conditional_edges("critique", should_continue)
        app = workflow.compile(checkpointer=memory)
        _vprint("No-search graph compiled.")

        

## 5. Batch Evaluate Compounds via No-Search Agent
Iterate through the sampled compounds, run the no-search agent workflow, and collect synthesized claims.

In [5]:
def _prepare_directive(prompt: str) -> str:
    requested_labels: List[str] = []
    match = re.search(r"(?i)answer:\s*(.*)", prompt)
    directive = prompt
    if match:
        raw_segment = match.group(1)
        candidates = [label.strip() for label in re.findall(r'"([^\"]+)"', raw_segment)]
        for label in candidates:
            if label and label not in requested_labels:
                requested_labels.append(label)
        if requested_labels:
            label_text = ", ".join(f'"{label}"' for label in requested_labels)
            directive = (
                f"{prompt}\n\nInstruction: ensure synthesis.claim exactly matches one of the user-specified answer options: {label_text}. "
                "Select the option best supported by your internal knowledge."
            )
    return directive


def run_no_search_agent(question: str, thread_tag: str | None = None) -> dict:
    _ensure_app()
    directive = _prepare_directive(question)
    init_message = HumanMessage(content=directive)
    init_state = {
        "messages": [init_message],
        "original_query": question,
        "stop_history": [],
        "consecutive_no_new_evidence": 0,
        "consecutive_stagnant_claim": 0,
        "consecutive_tool_unavailable": 0,
        "last_critique_signature": None,
        "consecutive_clarify_without_progress": 0,
    }

    config = {
        "configurable": {"thread_id": thread_tag or f"no_search_{uuid.uuid4().hex[:8]}"},
        "recursion_limit": MAX_ITERATIONS,
    }

    last_event_state = None
    
    # Wrap the streaming call in a try...except block to catch the recursion error
    try:
        for event in app.stream(init_state, config=config):
            for value in event.values():
                last_event_state = value
                if value and value.get("messages"):
                    try:
                        _vprint(value["messages"][-1].content)
                    except Exception:
                        _vprint(str(value))
    except GraphRecursionError:
        # This block executes when the recursion limit is hit.
        # It allows the function to continue and process the last available state.
        _vprint(f"--- GraphRecursionError: Recursion limit of {MAX_ITERATIONS} reached. Processing last state. ---")

    # Initialize a default result structure
    result = {
        "claim": "Unclear",
        "confidence": 0.0,
        "critique_json": {},
    }

    # This part of the code now runs even if a GraphRecursionError occurred
    if last_event_state and last_event_state.get("messages"):
        result["claim"] = determine_final_claim(last_event_state["messages"])
        result["confidence"] = determine_confidence_score(last_event_state["messages"])
        
        # Safely access the last message for critique JSON extraction
        if last_event_state["messages"]:
            critique_msg = last_event_state["messages"][-1]
            critique_raw = getattr(critique_msg, "content", "") or ""
            try:
                result["critique_json"] = json.loads(critique_raw)
            except Exception:
                match_json = re.search(r"\{[\s\S]*?\}", critique_raw)
                if match_json:
                    try:
                        result["critique_json"] = json.loads(match_json.group(0))
                    except Exception:
                        result["critique_json"] = {"error": "Failed to parse final JSON blob"}
                else:
                    result["critique_json"] = {"error": "No JSON found in the final message"}

    return result


batch_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
batch_dir = RESULTS_DIR / f"no_search_run_{batch_timestamp}"
batch_dir.mkdir(parents=True, exist_ok=True)

results: List[dict] = []
critique_snapshots: List[dict] = []

for idx, row in enumerate(tqdm(df_sampled.itertuples(index=False), total=len(df_sampled), desc="No-search evaluation")):
    compound_name = row.Name
    query = (
        f"Is the compound '{compound_name}' an endogenous metabolite from human? "
        "Answer \"Endogenous\" or \"Exogenous\"."
    )
    thread_id = f"no_search_{batch_timestamp}_{idx:03d}_{uuid.uuid4().hex[:6]}"
    agent_result = run_no_search_agent(query, thread_id)
    results.append({
        "Name": compound_name,
        "AgentClaim": agent_result.get("claim", "Unclear"),
        "AgentConfidence": float(agent_result.get("confidence") or 0.0),
    })
    critique_snapshots.append(agent_result.get("critique_json", {}))

results_df = pd.DataFrame(results)
results_df.head()

No-search evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

No-search graph compiled.
-----No-Search Node Invoked-----
```json
{
  "synthesis": {
    "claim": "Endogenous",
    "explanation": "The compound 'PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))' is a phosphatidylcholine (PC) with a plasmalogen (O-alkyl) linkage and specific fatty acyl chains. Phosphatidylcholines, including plasmalogens, are major components of human cell membranes and are synthesized endogenously.",
    "confidence_score": 0.9,
    "limitations": "The exact physiological role or abundance of this specific PC species is not immediately recalled, but its general class is well-known."
  },
  "traceability": {
    "evidence": [
      {
        "id": "memo-1",
        "title": "Phosphatidylcholine structure and function",
        "where_found": "model_memory",
        "quote": "Phosphatidylcholines are the most abundant phospholipids in eukaryotic cell membranes and are synthesized endogenously.",
        "why_relevant": "This establishes that phosphatidylcholines are endogenous t

No-search evaluation:   2%|▏         | 1/50 [00:05<04:53,  5.98s/it]

[should_continue] last critique output: {
  "decision": "approve",
  "reasoning": "Answer: Endogenous. The explanation clearly states that phosphatidylcholines, including the specific compound in question, are synthesized endogenously in hu...
-----Agent Decided to Stop (history stop_reason=APPROVED)-----
{
  "decision": "approve",
  "reasoning": "Answer: Endogenous. The explanation clearly states that phosphatidylcholines, including the specific compound in question, are synthesized endogenously in humans, supported by relevant evidence from model memory.",
  "next_query": "",
  "quality_checks": {
    "json_shape_ok": true,
    "claim_justified": true,
    "human_specific": true,
    "identification_confidence_ok": true,
    "evidence_diverse_sources": true
  },
  "stop_reason": "APPROVED"
}
-----No-Search Node Invoked-----
```json
{
  "synthesis": {
    "claim": "Unclear",
    "explanation": "The provided compound name 'CL(14:0/i-14:0/18:0/23:0)' represents a cardiolipin molecule wi

No-search evaluation:   4%|▍         | 2/50 [01:18<35:57, 44.94s/it]

[should_continue] last critique output: {
  "decision": "clarify",
  "reasoning": "The claim is marked as 'Unclear', which does not provide a definitive answer of 'Endogenous' or 'Exogenous'. The explanation highlights the unusual nature of...
{
  "decision": "clarify",
  "reasoning": "The claim is marked as 'Unclear', which does not provide a definitive answer of 'Endogenous' or 'Exogenous'. The explanation highlights the unusual nature of the compound, suggesting it may not be endogenous, but lacks a clear conclusion.",
  "next_query": "Investigate specific studies on the presence of CL(14:0/i-14:0/18:0/23:0) in human tissues.",
  "quality_checks": {
    "json_shape_ok": true,
    "claim_justified": false,
    "human_specific": true,
    "identification_confidence_ok": false,
    "evidence_diverse_sources": true
  },
  "stop_reason": ""
}
--- GraphRecursionError: Recursion limit of 20 reached. Processing last state. ---
-----No-Search Node Invoked-----
```json
{
  "synthesis": {
    

... (93 outputs left out)

[should_continue] last critique output: {
  "decision": "approve",
  "reasoning": "Answer: Exogenous. The explanation provided clarifies that while triglycerides can be endogenous, the specific branched and odd-chain fatty acids present in ...
-----Agent Decided to Stop (history stop_reason=APPROVED)-----
{
  "decision": "approve",
  "reasoning": "Answer: Exogenous. The explanation provided clarifies that while triglycerides can be endogenous, the specific branched and odd-chain fatty acids present in the compound suggest a dietary origin, justifying the claim of exogenous status.",
  "next_query": "",
  "quality_checks": {
    "json_shape_ok": true,
    "claim_justified": true,
    "human_specific": true,
    "identification_confidence_ok": true,
    "evidence_diverse_sources": true
  },
  "stop_reason": "APPROVED"
}
-----No-Search Node Invoked-----
```json
{
  "synthesis": {
    "claim": "Unclear",
    "explanation": "The provided compound name 'CL(10:0/i-14:0/24:0/25:0)' represents 

No-search evaluation: 100%|██████████| 50/50 [21:50<00:00, 26.22s/it]

[should_continue] last critique output: {
  "decision": "approve",
  "reasoning": "Answer: Exogenous. The explanation clearly outlines that the fatty acid composition of the compound is atypical for human cardiolipin, supporting the claim o...
-----Agent Decided to Stop (history stop_reason=APPROVED)-----
{
  "decision": "approve",
  "reasoning": "Answer: Exogenous. The explanation clearly outlines that the fatty acid composition of the compound is atypical for human cardiolipin, supporting the claim of it being exogenous.",
  "next_query": "",
  "quality_checks": {
    "json_shape_ok": true,
    "claim_justified": true,
    "human_specific": true,
    "identification_confidence_ok": true,
    "evidence_diverse_sources": true
  },
  "stop_reason": "APPROVED"
}


,Name,AgentClaim,AgentConfidence
0,"PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))",Endogenous,0.90
1,CL(14:0/i-14:0/18:0/23:0),Unclear,0.30
2,TG(22:0/i-12:0/i-24:0),Exogenous,0.80
3,alpha-Linolenic acid,Exogenous,0.95
4,CL(a-13:0/i-17:0/18:0/22:0),Unclear,0.30


In [9]:
results_df

,Name,AgentClaim,AgentConfidence
0,"PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))",Endogenous,0.90
1,CL(14:0/i-14:0/18:0/23:0),Unclear,0.30
2,TG(22:0/i-12:0/i-24:0),Exogenous,0.80
3,alpha-Linolenic acid,Exogenous,0.95
4,CL(a-13:0/i-17:0/18:0/22:0),Unclear,0.30
5,Potassium,Endogenous,1.00
6,TG(i-24:0/21:0/a-13:0)[rac],Exogenous,0.80
7,TG(21:0/12:0/i-24:0),Exogenous,0.90
8,CL(15:0/19:0/20:0/20:0),Exogenous,0.80
9,CL(i-12:0/i-19:0/a-21:0/a-25:0)[rac],Exogenous,0.80


## 6. Compare No-Search Predictions Against Labels
Join predictions with ground-truth annotations, compute accuracy metrics, and persist outputs for downstream analysis.

In [10]:
comparison = df_sampled.merge(results_df, on="Name", how="left")
comparison["GroundTruthLabel"] = comparison["Endogenous"].map({"Yes": "Endogenous", "No": "Exogenous"})
comparison["AgentClaim"] = comparison["AgentClaim"].fillna("Unclear")
comparison["Correct"] = comparison.apply(
    lambda row: str(row["AgentClaim"]).strip().lower() == str(row["GroundTruthLabel"]).strip().lower(),
    axis=1,
 )

accuracy = float(comparison["Correct"].mean()) if len(comparison) else 0.0
print(f"No-search accuracy: {accuracy:.2%} ({comparison['Correct'].sum()}/{len(comparison)})")

results_output_path = batch_dir / "no_search_agent_predictions.csv"
comparison.to_csv(results_output_path, index=False)
print(f"Saved predictions to {results_output_path}")

critique_output_path = batch_dir / "no_search_critiques.ndjson"
with critique_output_path.open("w", encoding="utf-8") as handle:
    for payload in critique_snapshots:
        handle.write(json.dumps(payload) + "\n")
print(f"Saved critique snapshots to {critique_output_path}")

comparison.head()

No-search accuracy: 28.00% (14/50)
Saved predictions to /home/chen/proj/db_form_agent_system/no_search_runs/no_search_run_20251013_193411/no_search_agent_predictions.csv
Saved critique snapshots to /home/chen/proj/db_form_agent_system/no_search_runs/no_search_run_20251013_193411/no_search_critiques.ndjson


,Accession,Name,Description,Chemical_Formula,Average_Molecular_Weight,Monisotopic_Molecular_Weight,IUPAC_Name,Traditional_IUPAC,CAS_Registry_Number,SMILES,InChI,InChIKey,Synonyms,Ontology,Endogenous,AgentClaim,AgentConfidence,GroundTruthLabel,Correct
0,HMDB0013450,"PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))","PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z)) is a ph...",C52H94NO7P,876.2790,875.676791,"(2-{[(2R)-2-[(4Z,7Z,10Z,13Z,16Z,19Z)-docosa-4,...","(2-{[(2R)-2-[(4Z,7Z,10Z,13Z,16Z,19Z)-docosa-4,...",NaN,CCCCCCCCCCCCCCCCCCCCCCOC[C@]([H])(COP([O-])(=O...,InChI=1S/C52H94NO7P/c1-6-8-10-12-14-16-18-20-2...,YLLDQHGIGWZVQP-ITBHSSNISA-N,1-Behenyl-2-docosahexaenoyl-sn-glycero-3-phosp...,\n,Yes,Endogenous,0.90,Endogenous,True
1,HMDB0223787,CL(14:0/i-14:0/18:0/23:0),CL(14:0/i-14:0/18:0/23:0) is a cardiolipin (CL...,C78H152O17P2,1424.0050,1423.050477,[(2R)-2-hydroxy-3-({hydroxy[(2R)-3-(octadecano...,(2R)-2-hydroxy-3-{[hydroxy((2R)-3-(octadecanoy...,NaN,[H][C@@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCC...,InChI=1S/C78H152O17P2/c1-6-9-12-15-18-21-24-26...,QJGGXVYKMFFNAP-FXCIZWNUSA-N,[(2R)-2-Hydroxy-3-({hydroxy[(2R)-3-(octadecano...,\n,Yes,Unclear,0.30,Endogenous,False
2,HMDB0068522,TG(22:0/i-12:0/i-24:0),TG(22:0/i-12:0/i-24:0) belongs to the family o...,C61H118O6,947.6090,946.892842,(2S)-3-(docosanoyloxy)-2-[(10-methylundecanoyl...,(2S)-3-(docosanoyloxy)-2-[(10-methylundecanoyl...,NaN,[H][C@](COC(=O)CCCCCCCCCCCCCCCCCCCCC)(COC(=O)C...,InChI=1S/C61H118O6/c1-6-7-8-9-10-11-12-13-14-1...,JAWPDRHZOXOVGJ-XKJQNMSGSA-N,1-behenoyl-2-isododecanoyl-3-isotetracosanoyl-...,\n,Yes,Exogenous,0.80,Endogenous,False
3,HMDB0001388,alpha-Linolenic acid,alpha-Linolenic acid (ALA) is a polyunsaturate...,C18H30O2,278.4296,278.224580,"(9Z,12Z,15Z)-octadeca-9,12,15-trienoic acid",α linolenic acid,463-40-1,CC\C=C/C\C=C/C\C=C/CCCCCCCC(O)=O,InChI=1S/C18H30O2/c1-2-3-4-5-6-7-8-9-10-11-12-...,DTOSIQBPPRVQHS-PDBXOOCHSA-N,"(9,12,15)-Linolenic acid; (9Z,12Z,15Z)-Octadec...",\n,No,Exogenous,0.95,Exogenous,True
4,HMDB0219059,CL(a-13:0/i-17:0/18:0/22:0),CL(a-13:0/i-17:0/18:0/22:0) is a cardiolipin (...,C79H154O17P2,1438.0320,1437.066127,[(2R)-3-({[(2R)-2-(docosanoyloxy)-3-(octadecan...,(2R)-3-{[(2R)-2-(docosanoyloxy)-3-(octadecanoy...,NaN,[H][C@@](O)(COP(O)(=O)OC[C@@]([H])(COC(=O)CCCC...,InChI=1S/C79H154O17P2/c1-7-10-12-14-16-18-20-2...,BFRUSIFRUHYVKZ-QDFPNRTDSA-N,[(2R)-3-({[(2R)-2-(docosanoyloxy)-3-(octadecan...,\n,Yes,Unclear,0.30,Endogenous,False


In [12]:
comparison[["Name", "GroundTruthLabel", "AgentClaim", "AgentConfidence", "Correct"]]

,Name,GroundTruthLabel,AgentClaim,AgentConfidence,Correct
0,"PC(O-22:0/22:6(4Z,7Z,10Z,13Z,16Z,19Z))",Endogenous,Endogenous,0.90,True
1,CL(14:0/i-14:0/18:0/23:0),Endogenous,Unclear,0.30,False
2,TG(22:0/i-12:0/i-24:0),Endogenous,Exogenous,0.80,False
3,alpha-Linolenic acid,Exogenous,Exogenous,0.95,True
4,CL(a-13:0/i-17:0/18:0/22:0),Endogenous,Unclear,0.30,False
5,Potassium,Exogenous,Endogenous,1.00,False
6,TG(i-24:0/21:0/a-13:0)[rac],Endogenous,Exogenous,0.80,False
7,TG(21:0/12:0/i-24:0),Endogenous,Exogenous,0.90,False
8,CL(15:0/19:0/20:0/20:0),Endogenous,Exogenous,0.80,False
9,CL(i-12:0/i-19:0/a-21:0/a-25:0)[rac],Endogenous,Exogenous,0.80,False


In [13]:
import json

In [15]:
# take string and convert to json
string = """{"synthesis": {"claim": "Exogenous", "explanation": "Biotin is a water-soluble vitamin (B7) that is an important coenzyme for various metabolic processes. While it is crucial for human health, the body cannot synthesize it endogenously. Humans obtain biotin through dietary intake and supplements, as indicated by discussions of daily recommended intake and supplementation.", "confidence_score": 0.85, "limitations": "The search results primarily discuss biotin as a dietary intake and supplement, and its role as a vitamin, but do not explicitly state that humans cannot synthesize it. However, the context strongly implies an exogenous source."}, "traceability": {"original_query": "is biotin endogenous metabolite from human? Answer: \"Endogenous\" or \"Exogenous\"", "compound_detected": true, "primary_candidate": {"name": "5-[(3aS,4S,6aR)-2-oxo-1,3,3a,4,6,6a-hexahydrothieno[3,4-d]imidazol-4-yl]pentanoic acid", "pubchem_cid": "171548", "inchikey": null, "smiles": "C1C2C(C(S1)CCCCC(=O)O)NC(=O)N2", "formula": "C10H16N2O3S", "mass_da": 244.316}, "synonyms": ["Biotin, Pharmaceutical Secondary Standard; Certified Reference Material", "Gelfert, Biotin", "1swr", "BPBio1_000414", "FB59502", "hexahydro-2-oxo-[3as-(3alpha,4beta,6alpha)]-1H-Thieno[3,4-d]imidazole-4-pentanoic acid", "Tox21_113050_1", "BIOTIN [FCC]", "BIOTIN [USP MONOGRAPH]", "Vitamin Essentaial", "Hair Cream", "EN300-54173", "CHEBI:15956", "1stp", "Hair Strengthens", "5-((3aS,4S,6aR)-2-oxo-1,3,3a,4,6,6a-hexahydrothieno(3,4-d)imidazol-4-yl)pentanoic acid", "2'-Keto-3,4-imidazolido-2-tetrahydrothiophene-N-valeric acid", "-(+)-Biotin", "HAIRJOY EYEBROW SIGNATURE", "CCRIS 3932"], "queries": {"pubmed": "biotin endogenous human", "arxiv": "biotin biosynthesis human", "extra": "biotin metabolism human"}, "search_call": {"query_param_sent": "[\"biotin endogenous human\", \"biotin biosynthesis human\", \"biotin metabolism human\"]", "max_results": 25, "include_preprints": true}, "runs": [{"variant": "pubmed", "run_dir": "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001", "saved_files": ["/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1016s0188-4409(02)00399-5.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1007978-94-007-2199-9_1.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1007978-1-4939-9537-0_4.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1126science.287.5460.2007.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.7554eLife.54983.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1093jn129.2.485S.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1146annurev.nutr.22.121101.112819.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1016j.jnutbio.2005.03.021.json", "/home/chen/proj/db_form_agent_agent_system/searching_results/run_20251015_001001/10.1002bit.24605.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10846444.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1021ja044996f.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.21741381612823666170727144450.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1093ajcn56.2.427.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1093jn122.3.535.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1111j.1530-0277.1993.tb00801.x.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1126sciadv.ado4880.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1016j.jnutbio.2005.03.018.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1093jn132.5.887.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1016j.xpro.2024.103321.json", "/home/chen/proj/db_form_agent_system/searching_results/run_20251015_001001/10.1093jn130.2.335S.json"]}], "evidence": [{"id": "10.1016s0188-4409(02)00399-5:4", "title": "Biotin in metabolism and its relationship to human disease.", "where_found": "Abstract", "quote": "Biotin is widely used as a health supplement and for medical treatment in biotin-deficiency individuals. It is a water-soluble vitamin, also known as vitamin B7, vitamin H, or coenzyme R, which is an important coenzyme for carboxylation related to gluconeogenesis, fatty acid metabolism, and amino acid catabolism [4]. The daily recommended intake of biotin is at least 30 μg. Healthy volunteers who ingested 5, 10, and 20 mg of biotin had median (minimum-maximum) peak serum biotin concentrations of 41 (10–73), 91 (53–141), and 184 (80–355) ng/mL, respectively, 1 h after ingestion [5].", "why_relevant": "This snippet describes biotin as a vitamin and mentions its daily recommended intake and use as a supplement, implying an exogenous source.", "link": "https://doi.org/10.1016/s0188-4409(02)00399-5"}, {"id": "10.1093jn122.3.535:5", "title": "Biotin in human milk: methods, location, and chemical form.", "where_found": "Discussion", "quote": "Adverse effects in humans due to high-dose biotin supplementation (up to 20 mg/day) have not been observed [3, 22]. The usual biotin intake via food and supplements does not represent a health risk to humans [23].", "why_relevant": "This snippet discusses biotin intake via food and supplements, further supporting its exogenous nature.", "link": "https://doi.org/10.1093/jn/122.3.535"}, {"id": "10.1007978-94-007-2199-9_1:3", "title": "Biotin: biochemical, physiological and clinical aspects.", "where_found": "Introduction", "quote": ". The physiological importance of biotin is primarily highlighted by the pathological effects that occur with its deficiency, which can lead to a variety of symptoms affecting the nervous system, skin, and respiratory system [6,10].", "why_relevant": "This snippet highlights the importance of biotin and the consequences of its deficiency, which is characteristic of essential nutrients that must be obtained exogenously.", "link": "https://doi.org/10.1007/978-94-007-2199-9_1"}, {"id": "10.1093jn122.3.535:52", "title": "Biotin in human milk: methods, location, and chemical form.", "where_found": "Discussion", "quote": "Hence, despite potential beneficial effects from biotin on glucose and lipid metabolism [76–81] observed in vitro and in animal studies (e.g., increased expression of glucokinase [78, 82], increased insulin secretion, improved glucose tolerance [78] and increased total beta-cell area relative to pancreas area [78, 79]), no preventive effects of biotin intakes exceeding the current reference values can be actually assumed with regard to nutrition-related chronic diseases. However, EFSA has approved several Article 13 health claims for biotin with regard to the “maintenance of normal skin and mucous membranes”, “maintenance of normal hair”, “contribution to normal psychological functions” and “normal macronutrient metabolism” [83].", "why_relevant": "This snippet discusses biotin intake and its health claims, reinforcing the idea of it being obtained from external sources.", "link": "https://doi.org/10.1093/jn/122.3.535"}, {"id": "10.1093jn122.3.535:43", "title": "Biotin in human milk: methods, location, and chemical form.", "where_found": "Discussion", "quote": ". Experimental data on the biotin requirement of lactating women are not available. Considering an average biotin loss of approximately 4 µg/day due to lactation (see derivation for infants at the age of 0 to under 4 months), an estimated value for an adequate intake of biotin for lactating women of 45 µg/day is derived.", "why_relevant": "This snippet mentions biotin requirement and adequate intake, which are concepts associated with essential nutrients that are not endogenously produced.", "link": "https://doi.org/10.1093/jn/122.3.535"}]}, "next_actions": {"missing_pieces": ["Direct statement confirming human inability to synthesize biotin."], "proposed_queries": ["human biotin biosynthesis pathway", "biotin essential nutrient human"]}}"""
json_object = json.loads(string) 

print(type(json_object)) 
print(json_object["name"])

JSONDecodeError: Expecting ',' delimiter: line 1 column 740 (char 739)